In [16]:
print(123)

123


In [17]:
import sys
sys.path.insert(0, '..')
from agentic_rag.ingest import load_faq_data

documents = load_faq_data()

In [18]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [19]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [20]:
documents = documents_llm

In [21]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [22]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [23]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [24]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()
openai_client = OpenAI(api_key=os.getenv("SECRET_OPENAI_API_KEY"))

In [25]:
import json

user_prompt = json.dumps(doc)

In [26]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [27]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [28]:
response.output_parsed.questions

['Can I still join the course if I found it late?',
 'If I join late, can I still get a certificate?',
 'What do I need to do to qualify for the certificate after joining now?',
 'Is it too late to start this course and participate?',
 'Do I have to submit the project before submissions close to get the certificate?']

In [29]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [30]:
from evaluation_utils import llm_structured

In [31]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — am I still allowed to join and follow along?', 'If I join the course now, is there still a chance to get the certificate?', 'Do I have to finish and submit the project before submissions close to get certified?', 'Can I start the course after it has already begun, or is it too late to enroll?', 'What’s the deadline for project submission if I want to earn the certificate?']


In [32]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=307)

In [33]:
from evaluation_utils import calc_price

In [34]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00045000000000000004,
 'total_cost': 0.0006052500000000001}

In [35]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — am I still allowed to join and follow along?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, is there still a chance to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to finish and submit the project before submissions close to get certified?',
  'document': '74eb249bbf'},
 {'question': 'Can I start the course after it has already begun, or is it too late to enroll?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I want to earn the certificate?',
  'document': '74eb249bbf'}]

In [36]:
import pandas as pd

In [37]:
pd.DataFrame(records)

,question,document
0,I just found this course late — am I still all...,74eb249bbf
1,"If I join the course now, is there still a cha...",74eb249bbf
2,Do I have to finish and submit the project bef...,74eb249bbf
3,Can I start the course after it has already be...,74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf


In [38]:
from evaluation_utils import llm_structured_retry

In [39]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [40]:
generate_ground_truth(doc)

([{'question': 'I just found this course late — can I still enroll and follow along?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to start the course after it has already begun?',
   'document': '74eb249bbf'},
  {'question': 'If I join the course now, do I still get access to everything?',
   'document': '74eb249bbf'},
  {'question': 'Can I still participate if I discovered the course after it started?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do if I want a certificate after joining late?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=87, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=294))

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [42]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [43]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [44]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [45]:
ground_truth[10]

{'question': 'How do I join the office hours or live workshop if I’m a student and don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [46]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08758949999999999

In [47]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08758949999999999

In [48]:
df_ground_truth = pd.DataFrame(ground_truth)

In [49]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [50]:
len(df_ground_truth)

565